# Contrôle qualité du fichier propre (Jour 3, étape 5)

**But :** vérifier à l'œil que `clean/verbatims.parquet` (produit par `python -m etl.run_clean`) ne contient plus de HTML, de donnée personnelle ni de message hors sujet.

**Comment l'utiliser :** exécuter les cellules dans l'ordre (Shift + Entrée), puis lire les 20 verbatims à voix haute.

## 1. Charger le fichier propre et compter les restes suspects

In [1]:
import os, pandas as pd
from dotenv import load_dotenv
load_dotenv()                                   # lit le .env (nom du bucket, clés AWS)
pd.set_option("display.max_colwidth", None)     # affiche les textes en entier
df = pd.read_parquet(f"s3://{os.environ['S3_BUCKET']}/clean/verbatims.parquet")
print(len(df), "verbatims propres")
print("Contient encore un @ :", df["text"].str.contains("@").sum())
print("Contient encore une balise < > :", df["text"].str.contains("<[a-z]", regex=True).sum())

447 verbatims propres
Contient encore un @ : 1
Contient encore une balise < > : 0


## 2. Vérifications automatiques du schéma
Le schéma validé au J3 : 8 colonnes, identifiant unique, texte de 30 à 3 000 caractères, langue `en` ou `fr`.

In [2]:
colonnes_attendues = ["verbatim_id", "source", "text", "context", "author_id", "created_on", "ingested_on", "lang"]
print("Colonnes conformes au schéma :", list(df.columns) == colonnes_attendues)
print("Identifiants uniques :", df["verbatim_id"].is_unique)
print("Longueur min / max :", df["text"].str.len().min(), "/", df["text"].str.len().max())
print("Langues :", df["lang"].value_counts().to_dict())
print("Liens masqués [URL] :", df["text"].str.contains(r"\[URL\]").sum(),
      "| e-mails masqués [EMAIL] :", df["text"].str.contains(r"\[EMAIL\]").sum())
print("Période couverte :", df["created_on"].min().date(), "->", df["created_on"].max().date())

Colonnes conformes au schéma : True
Identifiants uniques : True
Longueur min / max : 49 / 2812
Langues : {'en': 447}
Liens masqués [URL] : 24 | e-mails masqués [EMAIL] : 0
Période couverte : 2026-09-26 -> 2026-11-25


## 3. Lecture à voix haute : 20 verbatims tirés au hasard

Chacun·e en lit 7. Pour chaque ligne, se demander :
1. Reste-t-il du **HTML** ?
2. Reste-t-il un **nom** ou un pseudo ?
3. Le message est-il **hors sujet** (pas d'IA au travail) ?

In [3]:
df.sample(20, random_state=1)[["text", "lang"]]

,text,lang
189,"I was a commission-based salesperson at a SaaS company for several years. Your cold-email outreach didn't work because it was probably also AI written. When I was selling, I would write super short, warm, personalized emails by hand that sounded like a human wrote them, and wouldn't use AI to clean it up. My response rate hovered between 0.5 and 1.5%. My coworkers would copy and paste Claude-written emails into Gmail, and their response rate was 0.005%-0.05%. My biggest question, and this is not rhetorical, I'm genuinely curious for your reply: what was the problem that led you to build this product? Did you have the problem yourself? Did you talk to current and former coworkers about whether they were also having this problem? The other people here are rightly pointing out that your description of your own product sounds AI-written and soulless. The way you overcome this is by passionately talking about the problem instead of the product :)",en
292,"I dont wanna hate on this particular project, and I fully assume the possibility of PEBKAC, but I feel like my golden era of AI development was sometime during when I had a chat window in VS Code, and asked AI to make edits. Since then my company has 2 subsequent AI transformations, during the first we went fully agentic, and agents started pushing in 10k LOC PRs after churning away at the problem for hours. Even then I felt a profound loss of control, but now we are trying to take the next step, and even automate more stuff, and remove other bottlenecks, like CI/CD or VCS or ticketing, to cope with pushing an order of magnitude more code even than previously. So I'm feeling the exponentials, also the profound loss of control, and not having the faintest idea of what's going on. Now I understand how people get AI psychosis. As for the results, they are.... not there's a lot of things going on and I suspect most of said complexity is incidental.",en
185,"Disclaimer: I work as a sec engineer and part of my job includes finding vulnerabilities. I think the effort is misplaced. We should be exploring ways to make our systems more resilient and I would expect those companies* to focus on that front. Vulnerability research is important, and we should aim to remove as many bugs as we can from our codebases but the pace of software development is increasing and so is the introduction of new bugs. * I would also expect an increase on relevant research regarding novel paradigms of securing systems.",en
31,"AI being only as good as the person driving it is becoming less and less true with each passing month. If I were to hire tomorrow a developer for my company The least I would focus on is his expertise in languages. It just doesn't seem to matter anymore how good you are at a language for the vast majority of software development cases. If I had to say, I think steering in the AI to achieve a certain task is one of the main broad skills you need to have, but what that means exactly is a little bit hard to define. It's less about knowing exactly what to tell the AI and more about knowing how to break down a problem, give it the right context, recognize when it’s going in the wrong direction, and keep it moving toward the outcome you actually want and doing all that in the most efficient way you can.",en
347,"My employer has been integrating agentic AI into every tool and updating the UI/UX to make the agent the primary interface. This applies everywhere, including people management tasks. It sucks. In the past, you could look at a web page, see some structure, understand process, etc. and intuitively see and understand what you're suppose to do. When you remove all of that in favor of a text box, you spend a lot of time asking questions and prompting to get information or accomplish tasks that would otherwise have taken a minute or two and when you're done, you still don't understand how the underlying system, process, or policy is and how it works.",en
232,"My conclusion is

## 4. Ce qu'on a trouvé (à compléter pendant la lecture)

| # | Défaut trouvé | Règle à ajouter dans `etl/transform.py` |
|---|---|---|
| 1 | | |

Chaque défaut = une nouvelle règle, puis relancer `python -m etl.run_clean` et noter la règle et sa raison dans le journal.